# 🎬 AGAR-RL V10 : Visionneuse HD & Inspecteur Diagnostique Drive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Albin0903/agario/blob/main/notebooks/eval_drive_models.ipynb)

Ce notebook permet en 1 clic d'analyser vos modèles sauvegardés sur Google Drive (`agario_rl_backup_v10`, `agario_rl_backup_v9`, `agario_rl_backup_v8`, etc.) :
1. **Inspection Diagnostique Approfondie** : Analyse mathématique des poids, probabilités de split en zone de frappe, esquive des prédateurs mortels vs calme face aux rivaux inoffensifs, et appétence aux proies.
2. **Génération Replay Vidéo HD** : Match fluide de 80 secondes (2400 steps @ 30 FPS) avec HUD dynamique, radar minimap et vecteurs d'action.
3. **Dashboard d'Évolution Multi-Checkpoints** : Visualisation 6 panneaux de la progression de masse, kills, précision de split, survie et FPS.


In [ ]:
# 1. Montage sécurisé de Google Drive
import os, sys
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except ImportError:
    pass

V10_DIR = '/content/drive/MyDrive/agario_rl_backup_v10'
V9_DIR = '/content/drive/MyDrive/agario_rl_backup_v9'
V8_DIR = '/content/drive/MyDrive/agario_rl_backup_v8'
V7_DIR = '/content/drive/MyDrive/agario_rl_backup_v7'
V6_DIR = '/content/drive/MyDrive/agario_rl_backup_v6'
V5_DIR = '/content/drive/MyDrive/agario_rl_backup_v5'
print('=' * 65)
print('✅ Google Drive connecté.')
print(f'📁 Dossier V10 : {V10_DIR} (Existe: {os.path.exists(V10_DIR)})')
print(f'📁 Dossier V9 : {V9_DIR} (Existe: {os.path.exists(V9_DIR)})')
print(f'📁 Dossier V8 : {V8_DIR} (Existe: {os.path.exists(V8_DIR)})')
print(f'📁 Dossier V7 : {V7_DIR} (Existe: {os.path.exists(V7_DIR)})')
print('=' * 65)


In [ ]:
# 2. Synchronisation Git & Dépendances
import os

if os.path.exists('.git'):
    !git fetch origin main
    !git reset --hard origin/main
elif os.path.exists('agario/.git'):
    %cd agario
    !git fetch origin main
    !git reset --hard origin/main
else:
    !git clone https://github.com/Albin0903/agario.git
    %cd agario

os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.environ.get('PYTHONPATH', '')}"
!pip uninstall -y -q gym 2>/dev/null || true
!pip install -q -r requirements.txt
!apt-get install -qq -y ffmpeg
print('✅ Code et dépendances synchronisés avec succès.')

In [ ]:
# 3. Détection du Dernier Modèle (V10 prioritaire, fallback sur V9, V8, V7, V6 puis V5)
import os, glob, re

def extract_step(path):
    fname = os.path.basename(path)
    if 'final' in fname:
        return 999_999_999
    m = re.search(r'step_(\d+)', fname)
    return int(m.group(1)) if m else 0

def get_best_in_dir(dir_path):
    if not os.path.exists(dir_path):
        return None
    zips = glob.glob(os.path.join(dir_path, '*.zip'))
    valid = [z for z in zips if os.path.getsize(z) > 1000 and not os.path.basename(z).startswith('._') and 'bc_pretrained' not in z]
    if not valid:
        return None
    valid.sort(key=extract_step, reverse=True)
    return valid[0]

search_paths = [
    '/content/drive/MyDrive/agario_rl_backup_v10',
    '/content/drive/MyDrive/agario_rl_backup_v9',
    '/content/drive/MyDrive/agario_rl_backup_v8',
    '/content/drive/MyDrive/agario_rl_backup_v7',
    '/content/drive/MyDrive/agario_rl_backup_v6',
    '/content/drive/MyDrive/agario_rl_backup_v5',
    'checkpoints/ppo',
    'checkpoints/self_play_pool'
]

LATEST_MODEL = None
for sp in search_paths:
    best = get_best_in_dir(sp)
    if best:
        LATEST_MODEL = best
        break

if not LATEST_MODEL:
    raise FileNotFoundError('❌ Aucun checkpoint .zip valide trouvé sur Google Drive.')

step_number = extract_step(LATEST_MODEL)

print('=' * 75)
print(f'🏆 DERNIER CHECKPOINT DÉTECTÉ : {os.path.basename(LATEST_MODEL)}')
print(f'📊 Palier : {step_number:,} steps' if step_number < 999_999_999 else '📊 Palier : FINAL')
print(f'📁 Emplacement : {LATEST_MODEL}')
print('=' * 75)


## 4. Inspection Diagnostique Complète du Réseau de Neurones
Évalue le comportement de la politique sur 4 scénarios tactiques contrôlés et mesure la distribution des actions sur un épisode complet.

In [ ]:
import os
from IPython.display import HTML, display
from base64 import b64encode

os.makedirs('recordings', exist_ok=True)
output_video = 'recordings/eval_match_v10.mp4'

!python src/inference/record_match.py \
    --model "{LATEST_MODEL}" \
    --output "{output_video}" \
    --steps 2400

# Sauvegarde sur Drive si le dossier V10 existe
if os.path.exists(output_video) and os.path.exists(V10_DIR):
    !cp "{output_video}" "{V10_DIR}/eval_match_v10.mp4"
    print(f'📁 Vidéo sauvegardée sur Google Drive : {V10_DIR}/eval_match_v10.mp4')

if os.path.exists(output_video):
    mp4_bytes = open(output_video, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
    display(HTML(f'''
    <video width="850" height="480" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
    print(f'Taille de la vidéo : {os.path.getsize(output_video) / 1_000_000:.1f} Mo')
else:
    print('⚠️ Vidéo non trouvée.')


## 5. Génération & Visualisation du Replay HD Vidéo (80 secondes à 30 FPS)
Génère la vidéo avec le HUD complet et l'affiche directement dans le notebook.

In [ ]:
import os
from IPython.display import HTML, display
from base64 import b64encode

os.makedirs('recordings', exist_ok=True)
output_video = 'recordings/eval_match_v9.mp4'

!python src/inference/record_match.py \
    --model "{LATEST_MODEL}" \
    --output "{output_video}" \
    --steps 2400

# Sauvegarde sur Drive si le dossier V9 existe
if os.path.exists(output_video) and os.path.exists(V9_DIR):
    !cp "{output_video}" "{V9_DIR}/eval_match_v9.mp4"
    print(f'📁 Vidéo sauvegardée sur Google Drive : {V9_DIR}/eval_match_v9.mp4')

if os.path.exists(output_video):
    mp4_bytes = open(output_video, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
    display(HTML(f'''
    <video width="850" height="480" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
    print(f'Taille de la vidéo : {os.path.getsize(output_video) / 1_000_000:.1f} Mo')
else:
    print('⚠️ Vidéo non trouvée.')


## 6. Courbes d'Évolution & Comparatif Multi-Checkpoints (SOTA Academic Benchmarking)
Évalue l'ensemble des checkpoints de la version active (GoBigger / AgarCL standard) et génère un tableau de bord 6 panneaux :
- **Évolution de la Masse** (Pic & Moyenne)
- **Efficacité de Combat** (Kills / 1 000 steps)
- **Discipline & Précision de Split** (% de splits ciblant une proie réelle vs vide)
- **Horizon de Survie** (Nombre moyen de pas en vie par épisode)
- **Contrôle de Fragmentation** (Nombre moyen et maximal de sous-cellules)
- **Débit & Latence Moteur** (FPS réels de bout en bout et temps par pas en millisecondes)

In [ ]:
# Évalue les checkpoints de la version en cours et trace le tableau de bord d'évolution
import os
from IPython.display import Image, display

target_backup_dir = os.path.dirname(LATEST_MODEL)
plot_output_path = 'recordings/evolution_benchmark.png'

print('=' * 80)
print(f'📈 Audit comparatif d\'évolution sur le dossier : {target_backup_dir}')
print('=' * 80)

!python src/analysis/model_comparator.py \
    --dir "{target_backup_dir}" \
    --max-models 8 \
    --steps 1200 \
    --output "{plot_output_path}"

if os.path.exists(plot_output_path):
    display(Image(filename=plot_output_path))
    if os.path.exists(target_backup_dir):
        !cp "{plot_output_path}" "{target_backup_dir}/evolution_benchmark.png"
        print(f'📁 Graphique archivé sur Google Drive dans : {target_backup_dir}/evolution_benchmark.png')
else:
    print('⚠️ Graphique non généré (nécessite au moins 1 checkpoint dans le dossier).')
